# 01 — Introduction à PySpark et la SparkUI

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tanouar/1to1code/blob/main/pySpark/notebooks/01_introduction_jobSpark.ipynb)

Ce notebook crée une **SparkSession**, ouvre la **SparkUI** et exécute 5 tests de complexité croissante.

> Gardez la **SparkUI ouverte dans un onglet à côté** de ce notebook pour observer chaque job en temps réel.


In [ ]:
# Installation + setup (Colab)
!pip install -q pyspark findspark
!git clone --filter=blob:none --sparse https://github.com/tanouar/1to1code.git -q \
  && cd 1to1code && git sparse-checkout set pySpark -q

import os, sys
PYSPARK_TOOLS = os.path.join('/content', '1to1code', 'pySpark')
if PYSPARK_TOOLS not in sys.path:
    sys.path.insert(0, PYSPARK_TOOLS)

from sparktools.colab_setup import setup_colab

spark, monitor, helper = setup_colab("01 - Introduction PySpark")


## TEST 1 — Comptage simple

On crée un petit DataFrame et on compte ses lignes.  
→ SparkUI : **1 job** doit apparaître dans l'onglet **Jobs**.


In [ ]:
data = [
    ("Alice",   25, "IT",      50000),
    ("Bob",     30, "RH",      45000),
    ("Charlie", 35, "IT",      60000),
    ("David",   28, "Finance", 55000),
]
df = spark.createDataFrame(data, ["nom", "age", "departement", "salaire"])

result = monitor.execute_and_monitor(
    lambda: df.count(),
    "TEST 1: Comptage simple"
)
print(f"Résultat : {result} lignes")


## TEST 2 — Agrégation par département

On groupe les données par département.  
→ SparkUI : observez les **2 Stages** dans ce job (Map + Reduce = 1 shuffle).


In [ ]:
monitor.execute_and_monitor(
    lambda: df.groupBy("departement").count().show(),
    "TEST 2: Agrégation par département"
)


## TEST 3 — Gros volume (10 millions de lignes)

On mesure la vitesse de Spark sur un volume important.  
→ SparkUI : regardez le nombre de **Tasks** exécutées en parallèle.


In [ ]:
result = monitor.execute_and_monitor(
    lambda: spark.range(0, 10_000_000).toDF("id").count(),
    "TEST 3: Génération et comptage de 10M lignes"
)
print(f"Résultat : {result:,} lignes")


## TEST 4 — Analyse d'un DataFrame avec SparkHelper


In [ ]:
from pyspark.sql.functions import rand

df_test = spark.range(0, 1000).toDF("id") \
               .withColumn("valeur", (rand() * 100).cast("int"))

monitor.execute_and_monitor(
    lambda: helper.show_dataframe_info(df_test, "DataFrame Test"),
    "TEST 4: Analyse DataFrame"
)


## TEST 5 — Analyse du partitionnement


In [ ]:
df_big = spark.range(0, 1_000_000).toDF("id")

monitor.execute_and_monitor(
    lambda: helper.get_partition_info(df_big),
    "TEST 5: Analyse du partitionnement"
)


## Historique des jobs

Résumé de tous les jobs exécutés.  
→ SparkUI : vérifiez que les noms correspondent bien à l'onglet **Jobs**.


In [ ]:
# ============================================
# HISTORIQUE COMPLET
# ============================================

monitor.show_history()

In [ ]:
last_job = monitor.get_last_job()
if last_job:
    print(f"Dernier job   : {last_job['name']}")
    print(f"Durée         : {last_job['duration']:.2f}s")
    print(f"Status        : {last_job['status']}")
    print(f"Spark Job IDs : {last_job['spark_job_ids']}")
